In [1]:

import pandas as pd
from wearable_affect.data import PROJECT_ROOT
from wearable_affect.models import make_logreg, make_lightgbm
from wearable_affect.personalisation import loso_predict_personalised
from wearable_affect.tracking import log_run

features = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "features.parquet")

In [2]:
summary = []
for minutes in [2, 5, 10]:
    for model_name, make_model in [("logreg", make_logreg), ("lightgbm", make_lightgbm)]:
        for normalise in [False, True]:
            preds = loso_predict_personalised(features, make_model, minutes, normalise)

            for threshold_name, threshold_column in [("fixed", None), ("personal", "personal_threshold")]:
                name = f"pers-{minutes}min-{model_name}-{'norm' if normalise else 'raw'}-{threshold_name}"
                results = log_run(
                    name, preds, make_model, features,
                    params={"features": "neurokit_v1", "calibration_min": minutes,
                            "normalise": normalise, "threshold": threshold_name},
                    threshold_column=threshold_column,
                )
                summary.append({"minutes": minutes, "model": model_name, "normalise": normalise,
                                "threshold": threshold_name,
                                **results[["macro_f1", "auroc"]].mean().round(3).to_dict()})

summary = pd.DataFrame(summary)
summary

,minutes,model,normalise,threshold,macro_f1,auroc
0,2,logreg,False,fixed,0.840,0.967
1,2,logreg,False,personal,0.725,0.967
2,2,logreg,True,fixed,0.882,0.972
3,2,logreg,True,personal,0.820,0.972
4,2,lightgbm,False,fixed,0.860,0.976
5,2,lightgbm,False,personal,0.760,0.976
6,2,lightgbm,True,fixed,0.878,0.976
7,2,lightgbm,True,personal,0.691,0.976
8,5,logreg,False,fixed,0.839,0.966
9,5,logreg,False,personal,0.791,0.966


In [3]:
summary.pivot_table(index=["minutes", "model"], columns=["normalise", "threshold"], values="macro_f1")

normalise         False           True          
threshold         fixed personal  fixed personal
minutes model                                   
2       lightgbm  0.860    0.760  0.878    0.691
        logreg    0.840    0.725  0.882    0.820
5       lightgbm  0.850    0.832  0.881    0.746
        logreg    0.839    0.791  0.896    0.857
10      lightgbm  0.858    0.845  0.869    0.800
        logreg    0.872    0.812  0.908    0.880

1. Baseline normalisation works

With the fixed 0.5 threshold, subtracting each person's calm baseline improves mean F1 in all six combinations of calibration length and model:

Calibration	Logistic regression	LightGBM
2 min	0.840 → 0.882	0.860 → 0.878
5 min	0.839 → 0.896	0.850 → 0.881
10 min	0.872 → 0.908	0.858 → 0.869

Three things stand out.

Logistic regression benefits most, by +0.04 to +0.06, and with normalisation it becomes the best model overall, ahead of LightGBM. That makes sense: a linear model draws one boundary through the feature space, so if each person's data is shifted by a different amount, no single boundary fits everyone. Removing the personal offset lines everyone up for it. Trees can partly work around offsets with their many splits, so they gain less. It's also a nice result for the write-up: the simplest model, personalised, beats the more complex generic one.

AUROC barely changes, about 0.97 throughout. So normalisation doesn't improve how well the model ranks windows within a person; it was already very good at that. What it fixes is the level: each person's probabilities now sit in the right place relative to 0.5. That's exactly the problem your threshold analysis identified, now solved without needing any stress labels from the new user.

More calibration time helps logistic regression steadily: 0.882 at 2 minutes, 0.896 at 5, 0.908 at 10. But compare only within a calibration length. The generic baseline also changes across lengths (0.840 → 0.872), because longer calibration removes more early baseline windows from scoring, which changes the test set. The fair comparison is always normalised vs. raw at the same length, which is why we built it that way.

2. The personal threshold doesn't work, and hurts everywhere

In every one of the 12 comparisons, the personal threshold is worse than the fixed one, sometimes badly (LightGBM with normalisation at 2 minutes: 0.878 → 0.691). We said in advance this was possible, and it's a genuine finding. There are two likely reasons.

The calibration minutes don't represent all of a person's non-stress states. The threshold is set from the first few minutes of quiet sitting. But "non-stress" also includes the amusement condition, which is aroused and positive, and the later parts of baseline. Those can easily produce higher stress probabilities than the first quiet minutes. A threshold set at "higher than 95% of your quietest moments" then flags many perfectly fine non-stress windows as stress: false alarms.

LightGBM's extreme probabilities make it worse. We saw that LightGBM gives calm windows probabilities like 0.0001. A threshold set just above such tiny values is extremely low, so almost anything slightly higher crosses it. That's why LightGBM suffers most.

The lesson is useful for a product: calm-only calibration is good for normalising features, but not for choosing a decision threshold. A threshold needs to know what non-stressful activation looks like, not just stillness. Tempting as it is, don't now try other quantiles until something works; that would be the forking-paths problem again. Report it as tested and rejected, with the reasons.

In [4]:
# per subject check
from wearable_affect.tracking import load_run_results

rows = []
for minutes in [2, 5, 10]:
    for model in ["logreg", "lightgbm"]:
        raw = load_run_results(f"pers-{minutes}min-{model}-raw-fixed")
        norm = load_run_results(f"pers-{minutes}min-{model}-norm-fixed")
        diff = norm["macro_f1"] - raw["macro_f1"]
        rows.append({"minutes": minutes, "model": model,
                     "better": (diff > 0).sum(), "worse": (diff < 0).sum(), "tied": (diff == 0).sum()})
pd.DataFrame(rows)

,minutes,model,better,worse,tied
0,2,logreg,8,7,0
1,2,lightgbm,9,5,1
2,5,logreg,9,5,1
3,5,lightgbm,10,4,1
4,10,logreg,7,7,1
5,10,lightgbm,8,4,3


In [ ]:
raw = load_run_results("pers-5min-logreg-raw-fixed")
norm = load_run_results("pers-5min-logreg-norm-fixed")
pd.DataFrame({"f1_raw": raw["macro_f1"], "f1_norm": norm["macro_f1"],
              "auroc_raw": raw["auroc"], "auroc_norm": norm["auroc"]}).round(3)

,f1_raw,f1_norm,auroc_raw,auroc_norm
subject_id,,,,
S2,0.665,1.000,0.973,1.000
S3,0.820,0.718,0.917,0.930
S4,0.980,0.943,0.999,1.000
S5,0.960,1.000,1.000,1.000
S6,0.943,0.945,0.971,1.000
S7,0.789,0.890,0.935,1.000
S8,0.786,1.000,1.000,1.000
S9,0.819,0.720,0.877,0.777
S10,0.897,1.000,0.999,1.000


: 